# Notebook 1: Merge and clean data

This notebook merges raw CBS crime counts with neighbourhood covariates (BU-level) and attaches neighbourhood geometry. It then cleans the merged tabular dataset for downstream EDA and preprocessing.

### Inputs
- `datasets/raw/raw_crime_2024.csv`
- `datasets/raw/raw_nbh_2024.xlsx`
- `datasets/raw/wijkenbuurten_2024.gpkg`

### Outputs
- `datasets/pre-processing/merged_crime_nbh_2024.csv`
- `datasets/pre-processing/merged_crime_nbh_2024.gpkg`
- `datasets/pre-processing/cleaned_crime_nbh_2024.csv`


### Importing packages and setting options

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

RAW_DIR = Path("datasets/raw")
PREPROC_DIR = Path("datasets/pre-processing")
PREPROC_DIR.mkdir(parents=True, exist_ok=True)

CRIME_CSV = RAW_DIR / "raw_crime_2024.csv"
NBH_XLSX = RAW_DIR / "raw_nbh_2024.xlsx"
CBS_GPKG = RAW_DIR / "wijkenbuurten_2024.gpkg"
CBS_GPKG_LAYER = "buurten"
CBS_GEO_KEY_COL = "buurtcode"

TARGET_RAW_COL = "GeregistreerdeMisdrijven_1"
KEY_COL_CRIME = "WijkenEnBuurten"
KEY_COL_NBH = "gwb_code_10"

RANDOM_STATE = 42  # for downstream notebooks


### Load raw inputs

In [2]:
raw_crime = pd.read_csv(
    CRIME_CSV,
    sep=";",
    low_memory=False,
    dtype={KEY_COL_CRIME: "string", "ID": "string"},
)
raw_nbh = pd.read_excel(NBH_XLSX)

for _geo_id in ("gwb_code_10", "gwb_code_8", "gwb_code"):
    if _geo_id in raw_nbh.columns:
        raw_nbh[_geo_id] = raw_nbh[_geo_id].astype("string")

print("raw_crime:", raw_crime.shape)
print("raw_nbh:", raw_nbh.shape)


raw_crime: (18498, 7)
raw_nbh: (18310, 126)


### Inspect region-level codes (GM/WK/BU/NL)


In [3]:
print("Crime columns:", list(raw_crime.columns))
print("NBH columns:", list(raw_nbh.columns))
print()

print("GM_crime:", raw_crime[KEY_COL_CRIME].astype(str).str.startswith("GM").sum())
print("WK_crime:", raw_crime[KEY_COL_CRIME].astype(str).str.startswith("WK").sum())
print("BU_crime:", raw_crime[KEY_COL_CRIME].astype(str).str.startswith("BU").sum())
print("NL_crime:", raw_crime[KEY_COL_CRIME].astype(str).str.startswith("NL").sum())
print("Total Crime Rows:", raw_crime.shape[0])
print()

print("GM_nbh:", raw_nbh[KEY_COL_NBH].astype(str).str.startswith("GM").sum())
print("WK_nbh:", raw_nbh[KEY_COL_NBH].astype(str).str.startswith("WK").sum())
print("BU_nbh:", raw_nbh[KEY_COL_NBH].astype(str).str.startswith("BU").sum())
print("NL_nbh:", raw_nbh[KEY_COL_NBH].astype(str).str.startswith("NL").sum())
print("Total NBH Rows:", raw_nbh.shape[0])


Crime columns: ['ID', 'SoortMisdrijf', 'WijkenEnBuurten', 'Perioden', 'GeregistreerdeMisdrijven_1', 'Gemeentenaam_2', 'SoortRegio_3']
NBH columns: ['gwb_code_10', 'gwb_code_8', 'regio', 'gm_naam', 'recs', 'gwb_code', 'ind_wbi', 'a_inw', 'a_man', 'a_vrouw', 'a_00_14', 'a_15_24', 'a_25_44', 'a_45_64', 'a_65_oo', 'a_ongeh', 'a_gehuwd', 'a_gesch', 'a_verwed', 'a_nl_all', 'a_eur_al', 'a_neu_al', 'a_geb_nl', 'a_geb_eu', 'a_geb_ne', 'a_gbl_eu', 'a_gbl_ne', 'a_geb', 'p_geb', 'a_ste', 'p_ste', 'a_hh', 'a_1p_hh', 'a_hh_z_k', 'a_hh_m_k', 'g_hhgro', 'bev_dich', 'a_woning', 'a_nb_won ', 'a_vastg', 'a_nb_vastg', 'g_wozbag', 'p_1gezw', 'p_1gezw_tw', 'p_1gezw_hw', 'p_1gezw_2w', 'p_1gezw_hvw', 'p_mgezw', 'p_leegsw', 'p_koopw', 'p_huurw', 'p_wcorpw', 'p_ov_hw', 'p_bj_me10', 'p_bj_mi10', 'g_ele', 'g_ele_tr', 'g_gas', 'p_stadsv', 'p_won_z_ag', 'p_won_m_ag ', 'p_won_zs', 'p_won_ev ', 'a_lp_pub', 'a_ons_po', 'a_ons_vovavo', 'a_ons_mbo', 'a_ons_hbo', 'a_ons_wo', 'a_opl_bvm', 'a_opl_hvm', 'a_opl_hw', 'a_arb_w

### Filter to BU-level neighbourhood rows

In [4]:
crime = raw_crime[~raw_crime[KEY_COL_CRIME].astype(str).str.strip().str.startswith(("GM", "WK", "NL"), na=False)].copy()
nbh = raw_nbh[~raw_nbh[KEY_COL_NBH].astype(str).str.strip().str.startswith(("GM", "WK", "NL"), na=False)].copy()

print("crime (BU-level):", crime.shape)
print("nbh (BU-level):", nbh.shape)


crime (BU-level): (14730, 7)
nbh (BU-level): (14574, 126)


### Standardize join keys and run merge audits


In [5]:
def clean_key(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\s+", "", regex=True)
        .str.upper()
        .replace({"NAN": np.nan, "NONE": np.nan})
    )

crime["merge_key"] = clean_key(crime[KEY_COL_CRIME])
nbh["merge_key"] = clean_key(nbh[KEY_COL_NBH])

print("Missing merge_key (crime):", float(crime["merge_key"].isna().mean()))
print("Missing merge_key (nbh):", float(nbh["merge_key"].isna().mean()))
print()

crime_dupe_mask = crime["merge_key"].duplicated(keep=False)
nbh_dupe_mask = nbh["merge_key"].duplicated(keep=False)

print("Duplicate merge_key rows (crime):", int(crime_dupe_mask.sum()))
print("Duplicate merge_key rows (nbh):", int(nbh_dupe_mask.sum()))

if crime_dupe_mask.any():
    display(crime.loc[crime_dupe_mask, [KEY_COL_CRIME, "merge_key"]].value_counts().head(20))
if nbh_dupe_mask.any():
    display(nbh.loc[nbh_dupe_mask, [KEY_COL_NBH, "merge_key"]].value_counts().head(20))


Missing merge_key (crime): 0.0
Missing merge_key (nbh): 0.0

Duplicate merge_key rows (crime): 0
Duplicate merge_key rows (nbh): 0


### Merge crime and neighbourhood covariates

In [6]:
merged = nbh.merge(
    crime.drop(columns=[KEY_COL_CRIME], errors="ignore"),
    on="merge_key",
    how="left",
    validate="one_to_one",
)

print("merged shape:", merged.shape)
match_rate = float(merged[TARGET_RAW_COL].notna().mean())
print(f"Match rate on {TARGET_RAW_COL}: {match_rate:.3%}")


merged shape: (14574, 133)
Match rate on GeregistreerdeMisdrijven_1: 99.451%


### Drop unmatched crime rows (missing registered crimes)

Rows where `GeregistreerdeMisdrijven_1` is missing are treated as **unmatched records** (not zero-crime neighbourhoods) and are dropped so downstream analysis uses only matched crime counts.

In [7]:
missing_mask = merged[TARGET_RAW_COL].isna()
print("Unmatched rows (missing crime):", int(missing_mask.sum()))

merged = merged.loc[~missing_mask].copy().reset_index(drop=True)
print("After dropping unmatched:", merged.shape)


Unmatched rows (missing crime): 80
After dropping unmatched: (14494, 133)


### Attach geometry and export merged artifacts

In [8]:
geo = gpd.read_file(CBS_GPKG, layer=CBS_GPKG_LAYER)
geo[CBS_GEO_KEY_COL] = geo[CBS_GEO_KEY_COL].astype("string")

geo_small = geo[[CBS_GEO_KEY_COL, "geometry"]].copy()
geo_small["merge_key"] = clean_key(geo_small[CBS_GEO_KEY_COL])

merged_geom = merged.merge(
    geo_small[["merge_key", "geometry"]],
    on="merge_key",
    how="left",
)
merged_gdf = gpd.GeoDataFrame(merged_geom, geometry="geometry", crs=geo.crs)

n = len(merged_gdf)
matched = int(merged_gdf.geometry.notna().sum())
print(f"Geometry rows matched: {matched} / {n} ({100 * matched / n:.2f}%)")

# Export merged artifacts
merged_csv_out = PREPROC_DIR / "merged_crime_nbh_2024.csv"
merged_gpkg_out = PREPROC_DIR / "merged_crime_nbh_2024.gpkg"

merged_csv_out.parent.mkdir(parents=True, exist_ok=True)
merged_gpkg_out.parent.mkdir(parents=True, exist_ok=True)

merged.to_csv(merged_csv_out, index=False)
merged_gdf.to_file(merged_gpkg_out, driver="GPKG")

print("Saved:")
print("-", merged_csv_out)
print("-", merged_gpkg_out)


Geometry rows matched: 14494 / 14494 (100.00%)
Saved:
- datasets/pre-processing/merged_crime_nbh_2024.csv
- datasets/pre-processing/merged_crime_nbh_2024.gpkg


### Clean merged tabular dataset


In [9]:
df = pd.read_csv(
    merged_csv_out,
    na_values=["."],
    skipinitialspace=True,
    low_memory=False,
    dtype={
        "gwb_code_10": "string",
        "gwb_code_8": "string",
        "gwb_code": "string",
        "merge_key": "string",
        "ID": "string",
    },
)
print("Loaded merged CSV:", df.shape)


Loaded merged CSV: (14494, 133)


In [10]:
# Convert comma-decimal columns consistently (single pass)
comma_decimal_cols = ["g_afs_hp", "g_afs_gs", "g_afs_kv", "g_afs_sc", "g_3km_sc"]
comma_decimal_cols = [c for c in comma_decimal_cols if c in df.columns]

for c in comma_decimal_cols:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")

print("Converted comma-decimal cols:", comma_decimal_cols)

Converted comma-decimal cols: ['g_afs_hp', 'g_afs_gs', 'g_afs_kv', 'g_afs_sc', 'g_3km_sc']


In [11]:
# Fix: report the columns that look numeric-like (previous notebook printed an empty list)
possible_numeric_obj_cols = []
for col in df.select_dtypes(include="object").columns:
    ser = df[col].astype(str).str.strip().str.replace(",", "", regex=False)
    is_numeric_like = ser.replace(".", "0", regex=False).str.replace("-", "", regex=False).str.isnumeric()
    if float(is_numeric_like.mean()) > 0.7:
        possible_numeric_obj_cols.append(col)

print("Object columns that look numeric-like (fraction > 0.7):")
print(possible_numeric_obj_cols)

Object columns that look numeric-like (fraction > 0.7):
[]


In [12]:
# Drop administrative / merge-only columns
DROP_COLS = [
    "gwb_code_8",
    "recs",
    "ind_wbi",
    "gwb_code",
    "merge_key",
    "ID",
    "SoortMisdrijf",
    "Perioden",
    "Gemeentenaam_2",
    "SoortRegio_3",
]

drop_present = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=drop_present)

print("Dropped columns:", drop_present)
print("Cleaned df:", df.shape)

Dropped columns: ['gwb_code_8', 'recs', 'ind_wbi', 'gwb_code', 'merge_key', 'ID', 'SoortMisdrijf', 'Perioden', 'Gemeentenaam_2', 'SoortRegio_3']
Cleaned df: (14494, 123)


### Export cleaned dataset

In [13]:
cleaned_out = PREPROC_DIR / "cleaned_crime_nbh_2024.csv"
cleaned_out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(cleaned_out, index=False)

print("Saved:")
print("-", cleaned_out)


Saved:
- datasets/pre-processing/cleaned_crime_nbh_2024.csv


### Outputs saved

- `datasets/pre-processing/merged_crime_nbh_2024.csv`
- `datasets/pre-processing/merged_crime_nbh_2024.gpkg`
- `datasets/pre-processing/cleaned_crime_nbh_2024.csv`
